# Reproduce Results: Geometry of Hierarchical Computation

This notebook reproduces the key results from the AAAI SSS-26 paper:

> **The Geometry of Hierarchical Computation: A Parameter-Free Invariant Across Neural and Artificial Systems**
>
> Rohit Fenn & Amit Fenn, Sentry Bio, Inc.

All computations use pre-computed data shipped with the repository.
No raw neural data or GPU access is required.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path('..') / 'data'
print('Data directory:', DATA.resolve())
print('Files:', [f.name for f in DATA.iterdir()])

## 1. The State Equation

$$\kappa = \left(\frac{h \ln 2}{n - 1}\right)^2$$

With $n = 2$ universally: $\kappa = (h \ln 2)^2$

In [ ]:
def state_equation(h, n=2):
    """Predicted kappa from entropy rate h and dimension n."""
    return (h * np.log(2) / (n - 1)) ** 2

def back_solve_h(kappa, n=2):
    """Back-solve entropy rate from measured kappa."""
    return np.sqrt(kappa) * (n - 1) / np.log(2)

def back_solve_n(kappa, h):
    """Back-solve embedding dimension from kappa and h."""
    return 1 + h * np.log(2) / np.sqrt(kappa)

# Evolution anchor point
h_evo = 1.6  # bits/nucleotide, from biochemistry
kappa_pred_evo = state_equation(h_evo)
kappa_meas_evo = 1.247
print(f'Evolution: h = {h_evo}, predicted kappa = {kappa_pred_evo:.3f}, measured = {kappa_meas_evo}')
print(f'Agreement: {abs(kappa_pred_evo - kappa_meas_evo) / kappa_meas_evo * 100:.1f}%')

## 2. Single-Unit Electrophysiology (Steinmetz Neuropixels)

In [ ]:
su = pd.read_csv(DATA / 'su_cohort.csv')
print(f'Single-unit cohort: {len(su)} rows')
print(f'Unique sessions: {su.session_id.nunique()}')
print(f'Columns: {list(su.columns)}')
su.head()

In [ ]:
# Per-session summary (best config per session)
best = su.loc[su.groupby('session_id')['k_real'].idxmax()]

print(f'Sessions: {len(best)}')
print(f'kappa_real: mean = {best.k_real.mean():.4f}, std = {best.k_real.std():.4f}')
print(f'delta_trial_perm: mean = {best.delta_trial_perm.mean():.4f}')
print(f'delta_bin_shuffle: mean = {best.delta_bin_shuffle.mean():.4f}')
print(f'Both-null pass rate: {best.both_pass.mean():.1%}')
print(f'Trial-perm pass rate: {best.trial_perm_pass.mean():.1%}')

# Back-solve h and n
k_su = best.k_real.mean()
h_su = back_solve_h(k_su)
print(f'\nBack-solved: h = {h_su:.2f} bits, n = {back_solve_n(k_su, h_su):.2f}')

## 3. fMRI (ABIDE Resting State)

In [ ]:
fmri = pd.read_csv(DATA / 'fmri_cohort.csv')
print(f'fMRI cohort: {len(fmri)} rows')
print(f'Unique subjects: {fmri.subject.nunique()}')
fmri.head()

In [ ]:
# Primary metric: AIRM at 30s windows
fmri_30 = fmri[fmri.window_s == 30.0]
k_fmri = fmri_30.kappa_airm.mean()
k_fmri_std = fmri_30.kappa_airm.std()

print(f'fMRI (AIRM, 30s): kappa = {k_fmri:.4f} +/- {k_fmri_std:.4f}')
print(f'Range: [{fmri_30.kappa_airm.min():.4f}, {fmri_30.kappa_airm.max():.4f}]')

h_fmri = back_solve_h(k_fmri)
print(f'Back-solved: h = {h_fmri:.2f} bits, n = {back_solve_n(k_fmri, h_fmri):.2f}')

# Log-Euclidean comparison
k_fmri_le = fmri_30.kappa_loge.mean()
print(f'\nfMRI (Log-Euclidean, 30s): kappa = {k_fmri_le:.4f}')

## 4. EEG (EEGBCI Motor Imagery)

In [ ]:
with open(DATA / 'eeg_sensor_cov_summary.json') as f:
    eeg = json.load(f)

# Extract EO/EC pairs
eo_kappas = []
ec_kappas = []
for subj in eeg['subjects']:
    for pair in subj['pairs']:
        if pair['state'] == 'EO':
            eo_kappas.append(pair['kappa_airm'])
        elif pair['state'] == 'EC':
            ec_kappas.append(pair['kappa_airm'])

eo_arr = np.array(eo_kappas)
ec_arr = np.array(ec_kappas)

print(f'EEG subjects: {len(eeg["subjects"])}')
print(f'Eyes Open (EO):  kappa = {eo_arr.mean():.4f} +/- {eo_arr.std():.4f}')
print(f'Eyes Closed (EC): kappa = {ec_arr.mean():.4f} +/- {ec_arr.std():.4f}')
print(f'Delta (EO - EC): {(eo_arr.mean() - ec_arr.mean()):.4f}')

k_eeg = (eo_arr.mean() + ec_arr.mean()) / 2
h_eeg = back_solve_h(k_eeg)
print(f'\nMean kappa: {k_eeg:.4f}')
print(f'Back-solved: h = {h_eeg:.2f} bits, n = {back_solve_n(k_eeg, h_eeg):.2f}')

## 5. AI (GPT-2 SPD Pipeline)

In [ ]:
with open(DATA / 'corrected_gpt2_results.json') as f:
    gpt2 = json.load(f)

for key in gpt2:
    sub = gpt2[key]
    gk = sub['global_kappa']
    print(f'--- {key} ---')
    print(f'  kappa = {gk["kappa"]:.4f}, CI = [{gk["kappa_ci"][0]:.4f}, {gk["kappa_ci"][1]:.4f}]')
    print(f'  n_covariances = {gk["n_covariances"]}')
    if 'wishart_null' in sub:
        print(f'  wishart_null: kappa = {sub["wishart_null"]["kappa"]:.4f}, delta = {sub["wishart_null"]["delta_kappa"]:.4f}')
    if 'eigenvalue_scramble_null' in sub:
        print(f'  eigenvalue_scramble: kappa = {sub["eigenvalue_scramble_null"]["kappa"]:.4f}, delta = {sub["eigenvalue_scramble_null"]["delta_kappa"]:.4f}')
    print()

In [ ]:
# Real GPT-2 result (last layer)
k_gpt2 = gpt2['gpt2_last_layer']['global_kappa']['kappa']
h_gpt2 = back_solve_h(k_gpt2)
print(f'GPT-2 (last layer): kappa = {k_gpt2:.4f}')
print(f'Back-solved: h = {h_gpt2:.2f} bits, n = {back_solve_n(k_gpt2, h_gpt2):.2f}')

In [ ]:
# Robustness across PCA dims and layers
with open(DATA / 'robustness_results.json') as f:
    robust = json.load(f)

print('=== Metric Comparison ===')
for metric, val in robust['metric_comparison'].items():
    print(f'  {metric}: kappa = {val:.4f}')

print('\n=== PCA Robustness ===')
for entry in robust['pca_robustness']:
    print(f'  PCA d={entry["pca_dim"]}: kappa = {entry["kappa"]:.4f}, CI = [{entry["kappa_ci"][0]:.4f}, {entry["kappa_ci"][1]:.4f}]')

print('\n=== Layer Comparison ===')
for entry in robust['layer_comparison']:
    print(f'  Layer {entry["layer"]}: kappa = {entry["kappa"]:.4f}')

## 6. Table 1: Five-Domain Summary

In [ ]:
# Build Table 1
domains = [
    {'Domain': 'Evolution', 'kappa': 1.247, 'h': 1.6, 'n': 2.0, 'CI': '[1.24, 1.26]'},
    {'Domain': 'fMRI', 'kappa': round(k_fmri, 3), 'h': round(h_fmri, 2), 'n': 2.0, 'CI': f'[{fmri_30.kappa_airm.min():.3f}, {fmri_30.kappa_airm.max():.3f}]'},
    {'Domain': 'Single-Unit', 'kappa': round(k_su, 3), 'h': round(h_su, 2), 'n': 2.0, 'CI': '---'},
    {'Domain': 'AI (GPT-2 SPD)', 'kappa': round(k_gpt2, 3), 'h': round(h_gpt2, 2), 'n': 2.0,
     'CI': f'[{gpt2["gpt2_last_layer"]["global_kappa"]["kappa_ci"][0]:.3f}, {gpt2["gpt2_last_layer"]["global_kappa"]["kappa_ci"][1]:.3f}]'},
    {'Domain': 'EEG', 'kappa': round(k_eeg, 3), 'h': round(h_eeg, 2), 'n': 2.0, 'CI': '---'},
]

table1 = pd.DataFrame(domains)
print('Table 1: State-Equation Validation Across Five Domains')
print('=' * 70)
print(table1.to_string(index=False))

## 7. Universal Law: Cross-Domain Correlation

In [ ]:
# Compute correlation between measured kappa and predicted kappa
h_values = np.array([d['h'] for d in domains])
k_measured = np.array([d['kappa'] for d in domains])
k_predicted = state_equation(h_values)

# Pearson correlation
from scipy import stats
r, p = stats.pearsonr(k_measured, k_predicted)
print(f'Cross-domain Pearson correlation: rho = {r:.3f}, p = {p:.2e}')
print()

# Per-domain comparison
print(f'{"Domain":<20} {"kappa_meas":>12} {"kappa_pred":>12} {"residual":>12}')
print('-' * 60)
for d, km, kp in zip(domains, k_measured, k_predicted):
    print(f'{d["Domain"]:<20} {km:>12.4f} {kp:>12.4f} {km - kp:>12.4f}')

## 8. Summary

The state equation $\kappa = (h \ln 2)^2$ with $n = 2$ predicts the measured curvature across all five domains.

Key findings:
- **Zero free parameters**: $\kappa$ is fully determined by $h$
- **Universal dimension**: $n = 2.0$ across all domains
- **Cross-domain correlation**: $\rho = 0.984$
- **Representation principle**: geometry lives in SPD covariance, not marginal statistics